# Modelo de Identificação de Fraudes

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import datetime as dt

from scipy.stats import kstest, normaltest, anderson

from sklearn.preprocessing import PowerTransformer, MinMaxScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn import tree, metrics
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.under_sampling import RandomUnderSampler, ClusterCentroids, NearMiss 
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.combine import SMOTEENN

from sklearn.metrics import (
    confusion_matrix, 
    accuracy_score, 
    precision_score, 
    recall_score,
    precision_recall_curve,
)

import warnings
from src.config import DADOS_MASTERCARD

warnings.filterwarnings('ignore')
sns.set_theme(palette="bright")

In [6]:
# Importando a base

transacoes = pd.read_csv(DADOS_MASTERCARD)

In [4]:
# Verificar se todos os valores podem ser convertidos para inteiro (sem valores decimais)

if (transacoes['Time'] == transacoes['Time'].astype(int)).all():
    
    # Se todos os valores são inteiros, realizar a conversão
    
    transacoes['Time'] = transacoes['Time'].astype(int)
   
else:
    print("Nem todos os valores na coluna 'Time' podem ser convertidos para inteiro.")

### Vamos ajustar a escala de duas colunas para testar o resultado
- As colunas Time e Amount têm valores max muito alto. Destoa do restante do dataset.

In [6]:
# Primeiro para a coluna Time

transacoes.Time = transacoes.Time / transacoes.Time.max()

# E então para a coluna Amount

transacoes.Amount = transacoes.Amount / transacoes.Amount.max()

In [7]:
# Separando X e y

X = transacoes.drop('Class',axis=1)
y = transacoes.Class

**Separando em treino e teste**

In [9]:
# Separando em treino e teste

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=0,stratify=y)

### Realizando o random undersampling

In [11]:
# Definindo o RandomUnderSampler

rus = RandomUnderSampler(random_state=42)

In [12]:
# Definindo a nova amostra

X_resRUS, y_resRUS = rus.fit_resample(X_train, y_train)

**<font color='blue'>Com KNN**

In [14]:
# Fazendo o fit

clf_KNN = KNeighborsClassifier(n_neighbors=3).fit(X_resRUS, y_resRUS)

In [15]:
# Fazendo a previsão

y_pred_KNN = clf_KNN.predict(X_test)

y_pred_proba_KNN = clf_KNN.predict_proba(X_test)[:,1]

In [16]:
# Visualizando a matriz confusão

confusion_matrix(y_test,y_pred_KNN)  

array([[91227,  2598],
       [   18,   144]], dtype=int64)

In [17]:
# Calculando a área sob a curva precisão x recall

precision_KNN,recall_KNN,thresholds_KNN = metrics.precision_recall_curve(y_test,y_pred_proba_KNN)

print(metrics.auc(recall_KNN, precision_KNN))

0.5534876929920411


**E novamente utilizar o GridSearchCV**

In [19]:
# Utilizando os parâmetros do KNN

parametros = {'n_neighbors': [3,5,7,9,11], 'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'], 'weights': ['uniform', 'distance']}

In [20]:
# Selecionando o KNN

KNN_GS = KNeighborsClassifier()

In [21]:
# Criando um novo classificador usando os parâmetros que escolhemos anteriormente

clf_KNN_GS = GridSearchCV(KNN_GS, parametros, scoring='recall')

In [22]:
# Fazendo o fit dos nossos dados

clf_KNN_GS = clf_KNN_GS.fit(X_resRUS, y_resRUS)

In [23]:
# Visualizando os melhores parâmetros definidos pelo GridSearchCV 

clf_KNN_GS.best_params_

{'algorithm': 'auto', 'n_neighbors': 3, 'weights': 'uniform'}

In [24]:
# Usando esse modelo para fazer as previsões

y_pred_KNN_GS = clf_KNN_GS.predict(X_test)

In [25]:
# Analisando a matriz confusão

confusion_matrix(y_test,y_pred_KNN_GS)  

array([[91227,  2598],
       [   18,   144]], dtype=int64)